In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)


In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)




In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)



In [ ]:
# 4. Print shape of one batch

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)


In [ ]:
# 5. Display sample images
# Display first 6 images in the batch
fig, axes = plt.subplots(1, 6, figsize=(12, 3))

for i in range(6):
    img = images[i]

    # If image has channel dimension, remove it for display
    if img.dim() == 3:
        img = img.squeeze(0)

    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Label: {labels[i].item()}")
    axes[i].axis('off')

plt.show()



In [ ]:
# Task 1: Write your model class here:import torch
class FourLayerNN(nn.Module):
    def __init__(self, input_size, hidden1, hidden2, hidden3, output_size):
        super(FourLayerNN, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden1)
        self.layer2 = nn.Linear(hidden1, hidden2)
        self.layer3 = nn.Linear(hidden2, hidden3)
        self.layer4 = nn.Linear(hidden3, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))
        x = self.layer4(x)  # Usually no activation on the output for regression
        return x



In [ ]:
# Task 2: Write your training loop here:
def train_loop(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)  # sum loss over batch
    avg_loss = total_loss / len(dataloader.dataset)
    return avg_loss


In [ ]:
# Task 3: Write your validation loop here:
def val_loop(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = loss_fn(outputs, y)
            total_loss += loss.item() * X.size(0)
    avg_loss = total_loss / len(dataloader.dataset)
    return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = 10    # Example input features
hidden1 = 64
hidden2 = 32
hidden3 = 16
output_size = 1    # Example output

model = FourLayerNN(input_size, hidden1, hidden2, hidden3, output_size).to(device)

loss_fn = nn.MSELoss()          # Mean Squared Error for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:
from torch.utils.data import TensorDataset
X_train = torch.randn(500, input_size)
y_train = torch.randn(500, output_size)
X_val = torch.randn(100, input_size)
y_val = torch.randn(100, output_size)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

train_loss = [0.9, 0.7, 0.5, 0.4, 0.35]
val_loss = [1.0, 0.8, 0.6, 0.55, 0.5]
epochs = range(1, len(train_loss)+1)

plt.figure(figsize=(8,5))
plt.plot(epochs, train_loss, 'o-', label='Training Loss')
plt.plot(epochs, val_loss, 's-', label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here: